In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ==========================================================
# GENERATE ERROR BANK V1 (Clause-Level) - FINAL
# ==========================================================
import pandas as pd
import os

# 1. Load S1 Results (from the files generated by AblationStudy_S1_8.ipynb)

REV5_PATH = "/content/drive/MyDrive/ablation_outputs/system_1_bm25/S1_bm25_rev5_results.csv"
REV4_PATH = "/content/drive/MyDrive/ablation_outputs/system_1_bm25/S1_bm25_rev4_results.csv"

# 2. Helper to extract failures
def get_failures(csv_path, version_label):
    if not os.path.exists(csv_path):
        print(f" Missing file: {csv_path}")
        return pd.DataFrame()

    df = pd.read_csv(csv_path)

    # Ensure rank is numeric, turn NaNs/None into 0
    df["rank"] = pd.to_numeric(df["rank"], errors='coerce').fillna(0).astype(int)

    # Keep if Rank > 1 OR Rank == 0
    # This is mathematically the same as "Rank != 1"
    failures = df[df["rank"] != 1].copy()

    failures["version"] = version_label
    return failures

# 3. Process
print(" Generating Error Bank from S1 Results...")
err_rev5 = get_failures(REV5_PATH, "rev5")
err_rev4 = get_failures(REV4_PATH, "rev4")

# 4. Combine
raw_bank = pd.concat([err_rev5, err_rev4], ignore_index=True)

# 5. Format Columns
# Create 'id' (1, 2, 3...)
raw_bank.reset_index(drop=True, inplace=True)
raw_bank["id"] = raw_bank.index + 1

# Create 'question_id'
# (S1 results don't have the original ID, so we use the new ID as a placeholder)
raw_bank["question_id"] = raw_bank["id"]

# Set Constants
raw_bank["framework"] = "NIST_SP_800-53"

# Empty columns for manual entry
raw_bank["failure_category"] = ""
raw_bank["label_rationale"] = ""

# gold_control_id -> control_id, rank -> bm25_rank
raw_bank.rename(columns={
    "gold_control_id": "control_id",
    "rank": "bm25_rank"
}, inplace=True)

# 6. Select Final Columns in Order
final_cols = [
    "id",
    "question_id",
    "question",
    "framework",
    "version",
    "control_id",
    "bm25_rank",
    "failure_category",
    "label_rationale" # The only new column (required for dissertation)
]

final_df = raw_bank[final_cols]

# 7. Save (Overwriting the old one to be the new standard)
save_path = "/content/drive/MyDrive/compliance_data/error_bank/error_bank_v1.csv"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
final_df.to_csv(save_path, index=False)

print(f" Generated Error Bank V1: {len(final_df)} queries.")
print(f"   - Rev 5 Failures: {len(err_rev5)}")
print(f"   - Rev 4 Failures: {len(err_rev4)}")
print(f"   - Saved to: {save_path}")
# '0' in bm25_rank means Not Found in Top 10.
print(" ACTION: Open in Excel. Label the failures.")

 Generating Error Bank from S1 Results...
 Generated Error Bank V1: 39 queries.
   - Rev 5 Failures: 27
   - Rev 4 Failures: 12
   - Saved to: /content/drive/MyDrive/compliance_data/error_bank/error_bank_v1.csv
 ACTION: Open in Excel. Label the failures. '0' in bm25_rank means Not Found in Top 10.
